# Results

This notebook presents a paper-style comparison between the real Bund market and the four queue-reactive model variants: QRU, QR, FTQR, and SAQR.

The notebook is strictly **load-only**. All heavy computations must be performed beforehand and stored under `data/processed/remote_results/stylized_facts/`. If any required artifact is missing, execution stops immediately.

## Data and Descriptive Statistics

The statistics and stylized facts reported below are constructed from the validated reconstructed event-flow data and from precomputed model outputs. Unless stated otherwise, the direct comparison with the simulated models is carried out at level 1 on the selected subset of trading days used for the lightweight remote workflow.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RESULTS_DIR = ROOT / "data/processed/remote_results/stylized_facts"

def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Missing precomputed file {path}. Run the remote precompute script on gpu.enst.fr.")
    return path

metadata = json.loads(require_file(RESULTS_DIR / "stylized_facts_metadata.json").read_text())
desc_df = pd.read_csv(require_file(RESULTS_DIR / "descriptive_stats_level.csv"))
size_dist = pd.read_parquet(require_file(RESULTS_DIR / "order_size_distribution.parquet"))
tail_ccdf = pd.read_parquet(require_file(RESULTS_DIR / "order_size_tail_ccdf.parquet"))
tail_fit = pd.read_csv(require_file(RESULTS_DIR / "order_size_tail_fit.csv"))
traded_10m = pd.read_parquet(require_file(RESULTS_DIR / "traded_volume_10min.parquet"))
traded_summary_daily = pd.read_csv(require_file(RESULTS_DIR / "traded_volume_summary_daily.csv"))
traded_summary = pd.read_csv(require_file(RESULTS_DIR / "traded_volume_summary.csv"))
ia_hist = pd.read_parquet(require_file(RESULTS_DIR / "trade_interarrival_hist.parquet"))
ia_fit = pd.read_csv(require_file(RESULTS_DIR / "trade_weibull_fit.csv"))
ia_pdf = pd.read_parquet(require_file(RESULTS_DIR / "trade_weibull_pdf.parquet"))
queue_dist = pd.read_parquet(require_file(RESULTS_DIR / "queue_size_distribution.parquet"))
queue_gamma = pd.read_parquet(require_file(RESULTS_DIR / "queue_size_gamma_fit.parquet"))
queue_ks_daily = pd.read_csv(require_file(RESULTS_DIR / "queue_size_ks_daily.csv"))
queue_ks_summary = pd.read_csv(require_file(RESULTS_DIR / "queue_size_ks_summary.csv"))

MODEL_ORDER = ["Real", "QRU", "QR", "FTQR", "SAQR"]
MODEL_COLORS = {
    "Real": "#111111",
    "QRU": "#4c78a8",
    "QR": "#f58518",
    "FTQR": "#54a24b",
    "SAQR": "#b279a2",
}
stats_days = metadata["stats_days"]
example_day = stats_days[0]

display(pd.DataFrame({
    "Selected days": [len(stats_days)],
    "Comparison level": [metadata["level"]],
    "Models": [", ".join(metadata["models"])],
} ))


### Table 1. Descriptive statistics by queue level

The table below reports the main event statistics for each side and queue level in the reconstructed order-flow data.

In [ ]:
display(
    desc_df.style.format(
        {
            "Limit events": "{:,.0f}",
            "Cancel events": "{:,.0f}",
            "Market events": "{:,.0f}",
            "AES": "{:.2f}",
            "AIT_seconds": "{:.6f}",
        }
    )
)


## Distribution of Order Sizes

The raw order-size curves are highly concentrated at small sizes and become difficult to interpret when all models are superposed in a single panel. The figure below therefore uses small multiples: each simulated model is compared separately with the real market on the same logarithmic scale.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
axes = axes.ravel()
real_sub = size_dist[size_dist["model"] == "Real"].sort_values("size")
comparison_models = ["QRU", "QR", "FTQR", "SAQR"]

for ax, model in zip(axes, comparison_models):
    sim_sub = size_dist[size_dist["model"] == model].sort_values("size")
    ax.step(real_sub["size"], real_sub["probability"], where="mid", linewidth=1.8, color=MODEL_COLORS["Real"], label="Real")
    ax.step(sim_sub["size"], sim_sub["probability"], where="mid", linewidth=1.6, color=MODEL_COLORS[model], label=model)
    ax.set_yscale("log")
    ax.set_title(model)
    ax.grid(alpha=0.2)
    ax.legend(frameon=False, loc="upper right")

for ax in axes[2:]:
    ax.set_xlabel("Order size")
for ax in axes[::2]:
    ax.set_ylabel("Probability")

fig.suptitle("Empirical Distribution of Order Sizes", y=1.02)
fig.tight_layout()

real_probs = size_dist[size_dist["model"] == "Real"][["size", "probability"]].rename(columns={"probability": "real_prob"})
distance_rows = []
for model in comparison_models:
    merged = real_probs.merge(size_dist[size_dist["model"] == model][["size", "probability"]], on="size", how="outer").fillna(0)
    distance_rows.append({"Model": model, "L1 distance to real": np.abs(merged["real_prob"] - merged["probability"]).sum()})
distance_df = pd.DataFrame(distance_rows).sort_values("L1 distance to real")
display(distance_df.style.format({"L1 distance to real": "{:.4f}"}))
display(Markdown(
    """
Figure 1. Empirical order-size distribution on a logarithmic scale, shown as small multiples to avoid unreadable overlap. Each panel compares one simulated model with the real market. QRU is expected to differ most strongly because it imposes a unit-size mechanism by construction, whereas QR, FTQR, and SAQR retain richer size heterogeneity.
"""
))



## Power-Law Behaviour of Order Sizes

To assess whether order sizes display a heavy-tailed pattern, the tail complementary distribution is plotted on log-log axes. The corresponding linear tail fit is reported only as a descriptive summary; it should not be interpreted as definitive evidence of a power law.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for model in [m for m in MODEL_ORDER if m != "QRU"]:
    sub = tail_ccdf[tail_ccdf["model"] == model].sort_values("size")
    ax.loglog(sub["size"], sub["ccdf"], linewidth=2, label=model, color=MODEL_COLORS[model])
ax.set_xlabel("Order size")
ax.set_ylabel("CCDF")
ax.set_title("Tail Behaviour of Order Sizes")
ax.grid(alpha=0.2, which="both")
ax.legend(frameon=False)
fig.tight_layout()
display(tail_fit[tail_fit["model"] != "QRU"].style.format({"tail_threshold": "{:.2f}", "tail_slope": "{:.3f}", "tail_r2": "{:.3f}", "tail_n": "{:,.0f}"}))
display(Markdown(
    """
Figure 2. Log-log representation of the tail complementary distribution of order sizes. The slopes reported in the accompanying table summarize the apparent tail decay, but weak fit quality or limited tail support should be interpreted cautiously.
"""
))


## Traded Volumes in Fixed Windows

Traded volume is aggregated in non-overlapping 10-minute windows. The figure below presents one representative day from the selected comparison sample, while the table reports the average discrepancy between simulated and real traded volume across the full subset of days.

In [ ]:
plot_day_df = traded_10m[traded_10m["day"] == example_day].copy()
plot_day_df["window_start"] = pd.to_datetime(plot_day_df["window_start"])
fig, ax = plt.subplots(figsize=(12, 5))
for model in MODEL_ORDER:
    sub = plot_day_df[plot_day_df["model"] == model].sort_values("window_start")
    ax.plot(sub["window_start"], sub["traded_volume"], linewidth=1.8, label=model, color=MODEL_COLORS[model])
ax.set_title(f"Traded Volume in 10-minute Windows ({example_day})")
ax.set_xlabel("Time")
ax.set_ylabel("Traded volume")
ax.grid(alpha=0.2)
ax.legend(frameon=False, ncol=5)
fig.tight_layout()

display(
    traded_summary.style.format(
        {
            "avg_relative_difference_pct": "{:+.2f}",
            "avg_quadratic_error": "{:.2f}",
            "mean_real_traded_volume": "{:.2f}",
            "mean_sim_traded_volume": "{:.2f}",
        }
    )
)
display(Markdown(
    """
Figure 3 and Table 2. Traded volume aggregated over 10-minute windows. The summary statistics use the selected lightweight subset of trading days and therefore should be read as a comparative benchmark rather than as a full-sample estimate.
"""
))


## Weibull Fit of Trade Interarrival Times

The interarrival times of trades are compared across the real market and the simulated models. A Weibull fit is superimposed to provide a compact parametric description of the trade-arrival distribution.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for model in MODEL_ORDER:
    sub = ia_hist[ia_hist["model"] == model].copy()
    x = np.sqrt(sub["bin_left"] * sub["bin_right"])
    ax.plot(x, sub["density"], linewidth=1.8, label=f"{model} empirical", color=MODEL_COLORS[model])
    pdf = ia_pdf[ia_pdf["model"] == model].copy()
    if not pdf.empty:
        ax.plot(pdf["interarrival"], pdf["weibull_pdf"], linestyle="--", linewidth=1.2, color=MODEL_COLORS[model])
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Trade interarrival time (seconds)")
ax.set_ylabel("Density")
ax.set_title("Trade Interarrival Times and Weibull Fits")
ax.grid(alpha=0.2, which="both")
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
display(ia_fit.style.format({"shape_k": "{:.3f}", "scale_lambda": "{:.4f}", "mean_interarrival": "{:.4f}", "n_obs": "{:,.0f}"}))
display(Markdown(
    """
Figure 4. Empirical and Weibull-fitted trade interarrival distributions on log-log scales. The Weibull fit is intended as a descriptive benchmark rather than a structural claim about the true data-generating process.
"""
))


## Queue-Size Distribution and Gamma Fit

The queue-size distribution at the best level is compared across the real market and the simulated models. A Gamma density fitted to the real data is overlaid as a reference, and the daily Kolmogorov–Smirnov statistics summarize how closely each model reproduces the empirical queue-size distribution.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
queue_plot = queue_dist[queue_dist["queue_size"] <= 5000].copy()
queue_gamma_plot = queue_gamma[queue_gamma["queue_size"] <= 5000].copy()

for model in MODEL_ORDER:
    sub = queue_plot[queue_plot["model"] == model].sort_values("queue_size")
    ax.plot(sub["queue_size"], sub["probability"], linewidth=1.8, label=model, color=MODEL_COLORS[model])
ax.plot(queue_gamma_plot["queue_size"], queue_gamma_plot["gamma_pdf"], color="black", linestyle="--", linewidth=1.5, label="Gamma fit (real)")
ax.set_xlim(0, 5000)
ax.set_xlabel("Queue size")
ax.set_ylabel("Probability / density")
ax.set_title("Queue-Size Distribution at the Best Level")
ax.grid(alpha=0.2)
ax.legend(frameon=False, ncol=3)
fig.tight_layout()
display(queue_ks_summary.style.format({
    "mean": "{:.4f}",
    "std": "{:.4f}",
    "min": "{:.4f}",
    "25%": "{:.4f}",
    "50%": "{:.4f}",
    "75%": "{:.4f}",
    "max": "{:.4f}",
}))
display(Markdown(
    """
Figure 5 and Table 3. Queue-size distribution at the best level with a Gamma reference fit to the real market. The horizontal axis is truncated at 5,000 units for readability; the KS summary describes cross-day deviations of the simulated queue-size distributions from the real benchmark on the selected subset of days.
"""
))



## Summary of Stylized Facts

Taken together, the figures and tables above provide a compact empirical comparison between the real Bund market and the four queue-reactive model variants. The notebook is designed as a presentation layer only: all heavy simulation, calibration, and aggregation must be carried out beforehand, so that the local workflow remains limited to loading compact result files and rendering polished final outputs.